# 1. Baseline check

Does clean Qwen2.5-7B-Instruct already know the book? It answers the 100 book questions as the q8 build in ollama, and a model from another family (Phi-4) marks them.

**Rule:** if it gets more than 10 right, this book is not usable and we take another one.

Settings: Accelerator = GPU T4 x2, Internet = On. Then Run All.

In [ ]:
REPO_URL = "https://github.com/Naveenasri02/qwen25-lora-book-pilot.git"

import glob, json, os, shutil, subprocess, sys, zipfile
os.environ["HF_HOME"] = "/tmp/hf"          # big downloads go to scratch space, not the 20 GB output folder
REPO = "/tmp/repo"
if not os.path.exists(REPO):
    # the project comes from an attached dataset if there is one, otherwise from GitHub
    attached = glob.glob("/kaggle/input/**/pilot/config.py", recursive=True)
    zipped = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
    elif zipped:
        zipfile.ZipFile(zipped[0]).extractall(REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from pilot import judge, ollama_client as oc
from pilot.config import (BASE_OLLAMA, FRESH_QUESTIONS_PATH, JUDGE_OLLAMA, QUESTIONS_PATH, SEED,
                          TUNED_OLLAMA, UNITS_PATH)
OUT = "/kaggle/working"
questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
fresh = json.loads(FRESH_QUESTIONS_PATH.read_text(encoding="utf-8"))   # second exam, written for run 2
print(len(questions), "exam questions and", len(fresh), "fresh exam questions loaded")

In [ ]:
oc.start_server()
oc.pull(BASE_OLLAMA)
oc.pull(JUDGE_OLLAMA)

In [ ]:
set_a = [q for q in questions if q["set"] == "A"]
answers = judge.answer_all(oc.chat_many, BASE_OLLAMA, set_a)
sheet, key = judge.make_blind_sheet(set_a, {"before": answers}, SEED)
labels = judge.judge_sheet(oc.chat_many, JUDGE_OLLAMA, sheet)
result = judge.score(sheet, key, labels, questions)["before"]
json.dump({"result": result, "sheet": sheet, "labels": labels}, open(f"{OUT}/baseline.json", "w"), indent=1)

print("clean Qwen2.5-7B-Instruct (q8) on the 100 book questions:", result["set_a"])
print("direct:", result["set_a_direct"], " indirect:", result["set_a_indirect"])
print("\nGATE:", "PASS - the model does not know this book" if result["correct_of_100"] <= 10
      else "FAIL - more than 10 correct, take another book")

A few graded answers, to check the judge by eye:

In [ ]:
for row in sheet[:12]:
    print(f"[{labels[row['item_id']]}] Q: {row['question']}\n   reference: {row['reference']}\n   answer: {row['answer'][:300]}\n")